# Build a grounded RAG chat app on an Azure AI Search Basic index and prove the citations resolve

The docs team's 2,000-page product manual has been indexed in plain old keyword search for three years and users keep filing tickets that say "I cannot find anything in here." The product owner approves a six-week RAG pilot and the docs team picks a 50-document sample to start. You have one session to stand up an Azure AI Search Basic index with a vector plus semantic configuration, embed the corpus with `text-embedding-3-small`, build a RAG chat app that grounds GPT-4o-mini answers in the retrieved chunks, and confirm the citations actually point at real indexed chunks instead of being a polite hallucination.

What you will build:

- A Foundry hub plus project in `eastus`, plus a `gpt-4o-mini` deployment for chat and a `text-embedding-3-small` deployment for embeddings.
- An Azure AI Search Basic service. **This is the cost-critical resource in this lab.** Basic tier bills $73.73/month prorated hourly, about $0.10/hour, regardless of traffic.
- A search index with a 1536-dim vector field on HNSW plus a semantic configuration over `title` and `content`.
- A 50-chunk seeded corpus embedded with `text-embedding-3-small` and uploaded to the index.
- A RAG loop that embeds the user question, runs a hybrid plus semantic-ranked query for top-3 chunks, injects them into a GPT-4o-mini system prompt, and emits a structured answer with `citations: [{chunk_id, source_page}]`.

**Time.** Plan for about 75 minutes hands-on. Search service first-time create can take 3 to 8 minutes; budget up to 120 minutes total. The cleanup cell at the bottom tears every Azure resource down.

**Cost.** This is the lab where cleanup matters. Azure AI Search Basic costs ten cents an hour whether you are using it or not, and an overnight slip eats $2.45 of credit. A clean two-hour session lands around $0.20 to $0.30. GPT-4o-mini at $0.15/$0.60 per 1M tokens and `text-embedding-3-small` at $0.02 per 1M tokens both contribute fractions of a cent each. The whole point of the safety manifest and the atexit handler is to make the search-service teardown unforgettable. If you walk away from the notebook, the kernel timing out is not your friend. Run cleanup before you close the tab, every time.

This lab maps to AI-103 Domain 2: Implement generative AI and agentic solutions (33% of exam weight) and Domain 5: Implement information extraction solutions (13%).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the Azure SDKs this lab needs. Versions are
# pinned per LAB-CREATION-BLUEPRINT.md build rules.
# - azure-ai-projects 2.0 (March 2026) absorbed azure-ai-agents.
# - azure-search-documents is the data-plane SDK for index management and
#   document operations.
# - azure-mgmt-search is the management-plane SDK for Search service create
#   and delete.

!pip install --quiet labrun-checks==0.6.0 azure-identity>=1.15 azure-ai-projects==2.0.0 azure-mgmt-resource>=23.0.0 azure-mgmt-machinelearningservices>=1.0.0 azure-mgmt-cognitiveservices>=13.5.0 azure-mgmt-resourcegraph>=8.0.0 azure-mgmt-search>=9.1.0 azure-search-documents>=11.5.0 openai>=2.0.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT section 12.
# The search service name appends a 6-char random suffix because Search
# service names are globally unique.

import atexit
import getpass
import json
import random
import string
import time

from azure.identity import DefaultAzureCredential
from azure.core.exceptions import (
    HttpResponseError,
    ResourceNotFoundError,
    ClientAuthenticationError,
)
from azure.core.credentials import AzureKeyCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.machinelearningservices import MachineLearningServicesMgmtClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
from azure.mgmt.search import SearchManagementClient
from azure.mgmt.search.models import SearchService, Sku
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
)
from azure.search.documents.models import VectorizedQuery
from azure.ai.projects import AIProjectClient
from openai import AzureOpenAI

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "rag-with-azure-ai-search"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "eastus"

RESOURCE_GROUP = f"labrun-{LAB_ID}-rg"
HUB_NAME = f"labrun-{LAB_ID}-hub"
PROJECT_NAME = f"labrun-{LAB_ID}-project"
CHAT_DEPLOYMENT_NAME = f"labrun-{LAB_ID}-gpt4omini"
EMBED_DEPLOYMENT_NAME = f"labrun-{LAB_ID}-embed3s"
INDEX_NAME = "labrun-rag-index"

CHAT_MODEL_NAME = "gpt-4o-mini"
CHAT_MODEL_VERSION = "2024-07-18"
CHAT_MODEL_CAPACITY_TPM = 30  # sku.capacity unit is thousands of TPM
EMBED_MODEL_NAME = "text-embedding-3-small"
EMBED_MODEL_VERSION = "1"
EMBED_MODEL_CAPACITY_TPM = 30
EMBED_DIMENSIONS = 1536

# Search service names are globally unique; append a random suffix so two
# students in the same subscription do not collide on the same name.
_rand_suffix = "".join(random.choices(string.ascii_lowercase + string.digits, k=6))
SEARCH_SERVICE_NAME = f"labrun-rag-azure-search-{_rand_suffix}"

# Resolved during setup and Task 1.
SUBSCRIPTION_ID = None
AOAI_ACCOUNT_NAME = None
AOAI_ACCOUNT_ENDPOINT = None
PROJECT_ENDPOINT = None
SEARCH_ENDPOINT = None
AZURE_CREDENTIAL = None

# State populated by tasks for later checkpoints.
CORPUS_CHUNKS = []
QUERY_RESULTS = None
RAG_ANSWER = None
RAG_CITATIONS = None

# Pricing for wallet checks. Pricing claims must match
# https://azure.microsoft.com/en-us/pricing/details/cognitive-services/openai-service/
# and https://azure.microsoft.com/en-us/pricing/details/search/
CHAT_PRICE_PER_1M_INPUT_USD = 0.15
CHAT_PRICE_PER_1M_OUTPUT_USD = 0.60
EMBED_PRICE_PER_1M_TOKENS_USD = 0.02
SEARCH_BASIC_HOURLY_USD = 0.10

In [ ]:
# NBVAL_SKIP
# Register the labrun session and validate Azure credentials per
# LAB-CREATION-BLUEPRINT section 15. Search Basic is hourly-billed; the
# orphan scan in the next cell MUST run before any resource is created so a
# leftover service from a prior session does not silently accrue cost.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
subscription_id_input = getpass.getpass("AZURE_SUBSCRIPTION_ID: ").strip()
region_input = input(f"Azure region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported. This lab pins {REGION}.")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)
if not subscription_id_input:
    print("AZURE_SUBSCRIPTION_ID is required. Find yours with `az account show --query id -o tsv`.")
    raise SystemExit(1)

SUBSCRIPTION_ID = subscription_id_input

print("Asking DefaultAzureCredential to resolve your identity, hold on...")
try:
    AZURE_CREDENTIAL = DefaultAzureCredential()
    AZURE_CREDENTIAL.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as e:
    print("DefaultAzureCredential could not resolve a credential.")
    print("In Colab, run `!az login --use-device-code` in a fresh cell and re-run setup.")
    print(f"  Inner error: {e}")
    raise SystemExit(1)

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
try:
    next(iter(resource_client.resource_groups.list()))
except HttpResponseError as e:
    print(f"Could not list resource groups in subscription {SUBSCRIPTION_ID}: {e.message}")
    raise SystemExit(1)
except StopIteration:
    pass

AZURE_CREDENTIALS_BAG = {"subscription_id": SUBSCRIPTION_ID, "region": REGION}

print(f"Credentials validated. Subscription: {SUBSCRIPTION_ID}")
print(f"Region: {REGION}")
print(f"Resource group target: {RESOURCE_GROUP}")
print(f"Search service name (rand suffix): {SEARCH_SERVICE_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AZURE_CREDENTIALS_BAG)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and Resource Graph orphan scan.
# Critical resource: ai_search_service (Basic tier bills $73.73/month
# regardless of traffic, prorated hourly). Per RESOURCE-SAFETY-SPEC Rule 4,
# the orphan scan BLOCKS execution if any prior tagged resources exist.
#
# Reverse-creation order: search index first (dropped automatically with the
# service but the manifest is explicit), then the search service (critical),
# then the embedding deployment, the chat deployment, the project, hub, and
# resource group. The resource group delete is the cross-cutting safety net.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="search_index",
        id=INDEX_NAME,
        region=REGION,
        cli_delete_command=(
            f"az search index delete --service-name {SEARCH_SERVICE_NAME} "
            f"--name {INDEX_NAME} --yes"
        ),
    ),
    CleanupResource(
        type="ai_search_service",
        id=SEARCH_SERVICE_NAME,
        region=REGION,
        cli_delete_command=(
            f"az search service delete --resource-group {RESOURCE_GROUP} "
            f"--name {SEARCH_SERVICE_NAME} --yes"
        ),
    ),
    CleanupResource(
        type="aoai_deployment",
        id=EMBED_DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} --name <attached-aoai-account> "
            f"--deployment-name {EMBED_DEPLOYMENT_NAME}"
        ),
    ),
    CleanupResource(
        type="aoai_deployment",
        id=CHAT_DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} --name <attached-aoai-account> "
            f"--deployment-name {CHAT_DEPLOYMENT_NAME}"
        ),
    ),
    CleanupResource(
        type="foundry_project",
        id=PROJECT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {PROJECT_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="foundry_hub",
        id=HUB_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {HUB_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="resource_group",
        id=RESOURCE_GROUP,
        region=REGION,
        cli_delete_command=f"az group delete --name {RESOURCE_GROUP} --yes --no-wait",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    graph_client = ResourceGraphClient(AZURE_CREDENTIAL)
    query = (
        f"Resources | where tags['{LAB_TAG_KEY}'] == '{LAB_TAG_VALUE}' "
        f"| project id, name, type, location"
    )
    request = QueryRequest(subscriptions=[SUBSCRIPTION_ID], query=query)
    response = graph_client.resources(request)
    return [row.get("id", "") for row in (response.data or [])]


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run. Search Basic bills $0.10/hour")
    print("even when idle, so leaving an orphan service running costs $2.45/day.")
    print("Run the cleanup cell at the bottom of this notebook, or manually:")
    print(f"  az group delete --name {RESOURCE_GROUP} --yes --no-wait")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up Foundry, deploy chat plus embedding models, and create the Search Basic service

The Foundry stack is the same shape as Lab 01: hub, project, attached Azure OpenAI account, and two model deployments (this lab needs both `gpt-4o-mini` for chat and `text-embedding-3-small` for embeddings). The new piece is the Azure AI Search Basic service: a 2-to-8-minute provisioning step that creates an hourly-billed resource. The lab's whole cost-and-safety lesson lives here.

Build it in this order:

1. Create the resource group, hub, and project with the lab tag. Hub provisioning is async (4 to 6 minutes); project provisioning is async (2 to 3 minutes). The cell waits for both.
2. Discover the hub's attached Azure OpenAI account and deploy `gpt-4o-mini` and `text-embedding-3-small` against it. Both at Standard SKU.
3. Call `SearchManagementClient.services.begin_create_or_update` to provision the Search service at Basic tier in `eastus`, with the lab tag on the service itself. Wait for `provisioningState=Succeeded`. The validator REJECTS Free tier (cannot host the 50-doc embedded corpus) and Standard or higher (cost overkill for this workload).

**Why Basic and not Free.** The Free SKU has a 50 MB index cap and no vector profile support beyond a tiny preview, which is the wrong shape for production RAG. Basic is the smallest production-shaped tier and the one the AI-103 study guide assumes.

**Why this is the cost-critical resource.** Once `begin_create_or_update` completes, you are billing $0.10/hour. If you walk away and the kernel times out, the service keeps billing until you (or the resource-group atexit fallback, or you on the CLI tomorrow morning) delete it. Setup registers the service in `CLEANUP_MANIFEST` before the create even returns so the atexit handler can find it.

In [ ]:
# NBVAL_SKIP
# Task 1: Provision RG, hub, project, GPT-4o-mini and text-embedding-3-small
# deployments, then create the Search Basic service.

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
cs_client = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
search_mgmt_client = SearchManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

lab_tags = {LAB_TAG_KEY: LAB_TAG_VALUE}

# Resource group, hub, project. Same pattern as Lab 01.
resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP, {"location": REGION, "tags": lab_tags},
)
print(f"Resource group ready: {RESOURCE_GROUP}")

hub_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Hub",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": HUB_NAME, "public_network_access": "Enabled"},
}
print("Asking ARM to allocate a Foundry hub, hold on for about 4 to 6 minutes...")
hub_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, HUB_NAME, hub_payload,
).result()

project_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Project",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": PROJECT_NAME, "hub_resource_id": hub_workspace.id},
}
print("Watching the project workspace go through provisioning, 2 to 3 minutes more...")
project_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, PROJECT_NAME, project_payload,
).result()
PROJECT_ENDPOINT = project_workspace.properties.discovery_url

# Discover the attached Azure OpenAI account.
for conn in ml_client.connections.list(RESOURCE_GROUP, HUB_NAME):
    if (conn.properties.category or "").lower() == "azureopenai":
        target = conn.properties.target or ""
        if "/accounts/" in target:
            AOAI_ACCOUNT_NAME = target.split("/accounts/")[-1].split("/")[0]
            break
if not AOAI_ACCOUNT_NAME:
    print("Could not find an attached Azure OpenAI account on the hub.")
    raise SystemExit(1)

aoai_account = cs_client.accounts.get(RESOURCE_GROUP, AOAI_ACCOUNT_NAME)
AOAI_ACCOUNT_ENDPOINT = aoai_account.properties.endpoint
print(f"Attached Azure OpenAI account: {AOAI_ACCOUNT_NAME}")
print(f"Endpoint: {AOAI_ACCOUNT_ENDPOINT}")

# Deploy GPT-4o-mini and text-embedding-3-small.
chat_payload = {
    "sku": {"name": "Standard", "capacity": CHAT_MODEL_CAPACITY_TPM},
    "properties": {
        "model": {"format": "OpenAI", "name": CHAT_MODEL_NAME, "version": CHAT_MODEL_VERSION},
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}
print("Watching the chat deployment go through Succeeded purgatory...")
cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, CHAT_DEPLOYMENT_NAME, chat_payload,
).result()
print(f"GPT-4o-mini deployment ready: {CHAT_DEPLOYMENT_NAME}")

embed_payload = {
    "sku": {"name": "Standard", "capacity": EMBED_MODEL_CAPACITY_TPM},
    "properties": {
        "model": {"format": "OpenAI", "name": EMBED_MODEL_NAME, "version": EMBED_MODEL_VERSION},
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}
print("Same pattern for the embedding deployment...")
cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, EMBED_DEPLOYMENT_NAME, embed_payload,
).result()
print(f"text-embedding-3-small deployment ready: {EMBED_DEPLOYMENT_NAME}")

# Search Basic service. This is the cost-critical create. The validator
# rejects Free (50 MB cap, no real vector profile) and Standard or higher
# (cost overkill).
search_payload = SearchService(
    location=REGION,
    sku=Sku(name="basic"),
    tags=lab_tags,
    partition_count=1,
    replica_count=1,
    public_network_access="enabled",
)
print("Asking ARM to allocate an Azure AI Search Basic service, this takes 3 to 8 minutes...")
print(f"Once this returns, you are billing ${SEARCH_BASIC_HOURLY_USD:.2f}/hour until cleanup runs.")
# YOUR CODE: Start the search-service create via
# search_mgmt_client.services.begin_create_or_update(
#     RESOURCE_GROUP, SEARCH_SERVICE_NAME, search_payload,
# )
# and call .result() on the poller. Store the result in search_service.

SEARCH_ENDPOINT = f"https://{SEARCH_SERVICE_NAME}.search.windows.net"
print(f"Search service provisioned: {SEARCH_SERVICE_NAME}")
print(f"Endpoint: {SEARCH_ENDPOINT}")
print(f"SKU: {search_service.sku.name}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Search service exists at Basic tier in eastus with the lab
# tag, provisioningState=Succeeded. Rejects Free and Standard or higher.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        sm = SearchManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
        try:
            svc = sm.services.get(RESOURCE_GROUP, SEARCH_SERVICE_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Search service {SEARCH_SERVICE_NAME} does not exist in "
                    f"{RESOURCE_GROUP}. Did you call .result() on the poller?"
                ),
            )

        sku_name = (svc.sku.name or "").lower() if svc.sku else ""
        if sku_name != "basic":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Search SKU is {sku_name!r}, expected 'basic'. Free has a 50 MB "
                    f"cap and no real vector profile; Standard or higher is cost overkill "
                    f"for a 50-doc lab corpus."
                ),
            )
        if (svc.location or "").lower() != REGION:
            return CheckpointResult(
                status="fail",
                error_reason=f"Search location is {svc.location!r}, expected {REGION!r}.",
            )
        tags = svc.tags or {}
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Search service is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Found tags: {tags}. The orphan scan needs this tag to find the "
                    f"service in a future session."
                ),
            )
        prov = svc.provisioning_state
        if prov != "succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Search provisioningState is {prov!r}, expected 'succeeded'. "
                    f"The poller returned before provisioning completed."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names the field that is off: SKU, location, tag, or provisioning state. A common miss: passing the SKU as a string rather than a `Sku(name="basic")` model. If the call fails outright with `LocationNotAvailableForSku`, your subscription does not have Basic available in eastus; pick a different region from the supported list (and update REGION).

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `search_mgmt_client.services.begin_create_or_update(RESOURCE_GROUP, SEARCH_SERVICE_NAME, search_payload)`. The poller is async; call `.result()` and store in `search_service` before the next print. The `search_payload` variable already has `sku=Sku(name="basic")`, `location=eastus`, `tags=lab_tags`, and `partition_count=replica_count=1`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
search_service = search_mgmt_client.services.begin_create_or_update(
    RESOURCE_GROUP, SEARCH_SERVICE_NAME, search_payload,
).result()
```

</details>

**Wallet check.** The meter just started. The Search service has been alive for a few minutes, so you have spent roughly one cent on Search Basic ($0.10/hour x 0.1 hour). Foundry hub, project, and Standard deployments are free at zero traffic. Total damage so far: about a cent. Your morning coffee is still 400x ahead. Don't forget the cleanup cell.

## Task 2: Build the index with a 1536-dim vector field, an HNSW vector profile, and a semantic configuration

The index is where vector search, hybrid search, and the semantic ranker all live. You declare the field shape, the vector profile that controls ANN behaviour, and the semantic configuration that names which fields to feed the reranker.

Build it in this order:

1. Declare the fields. `id` is the key field. `title` and `content` are searchable strings (so BM25 keyword search works against them). `content_vector` is a `Collection(Edm.Single)` with `dimensions=1536` and `vector_search_profile_name` pointing at the HNSW profile you declare next. `source_page` is a filterable int.
2. Declare the vector profile: one `HnswAlgorithmConfiguration` named `hnsw-config`, one `VectorSearchProfile` named `vector-profile` that references it.
3. Declare the semantic configuration: one `SemanticConfiguration` named `default`, with `prioritized_fields` listing `title` as the title field and `content` as a content field.
4. Build the `SearchIndex` payload and call `create_or_update_index` on a `SearchIndexClient`.

**Why dimensions must equal 1536.** `text-embedding-3-small` returns 1536-dim vectors. The index field declares the vector size; a mismatch (1024 or 3072) fails query time, not at create time, which is a subtle trap. The validator rejects anything other than 1536.

**Why HNSW.** Hierarchical Navigable Small World is the default Azure AI Search vector index algorithm. It is fast on Basic tier; exhaustive KNN exists but is slow above ~10k vectors. For 50 chunks the difference is invisible.

In [ ]:
# NBVAL_SKIP
# Task 2: Build the index with vector + semantic configuration.

search_index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT, credential=AZURE_CREDENTIAL,
)

# Declare the vector profile and the semantic configuration.
hnsw_config = HnswAlgorithmConfiguration(name="hnsw-config")
vector_profile = VectorSearchProfile(
    name="vector-profile",
    algorithm_configuration_name="hnsw-config",
)
vector_search = VectorSearch(
    algorithms=[hnsw_config],
    profiles=[vector_profile],
)

semantic_config = SemanticConfiguration(
    name="default",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        content_fields=[SemanticField(field_name="content")],
    ),
)
semantic_search = SemanticSearch(configurations=[semantic_config])

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
    SearchableField(name="title", type=SearchFieldDataType.String),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBED_DIMENSIONS,
        vector_search_profile_name="vector-profile",
    ),
    SimpleField(
        name="source_page",
        type=SearchFieldDataType.Int32,
        filterable=True,
    ),
]

# YOUR CODE: Build the SearchIndex payload and call create_or_update_index.
# Construct index = SearchIndex(name=INDEX_NAME, fields=fields,
# vector_search=vector_search, semantic_search=semantic_search).
# Then call search_index_client.create_or_update_index(index).
# Store the returned index object in created_index.

print(f"Index ready: {created_index.name}")
print(f"  Fields: {[f.name for f in created_index.fields]}")
print(f"  Vector profiles: {[p.name for p in created_index.vector_search.profiles]}")
print(f"  Semantic configs: {[c.name for c in created_index.semantic_search.configurations]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Index has id, title, content, content_vector (1536 dims,
# vector profile set), source_page; an HNSW algorithm config; a semantic
# configuration naming title and content.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        sic = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=AZURE_CREDENTIAL)
        try:
            idx = sic.get_index(INDEX_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Index {INDEX_NAME} does not exist on {SEARCH_SERVICE_NAME}. "
                    f"Did create_or_update_index run?"
                ),
            )

        by_name = {f.name: f for f in idx.fields}
        for required in ("id", "title", "content", "content_vector", "source_page"):
            if required not in by_name:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Index is missing field {required!r}. Found: {sorted(by_name.keys())}"
                    ),
                )

        vec = by_name["content_vector"]
        if not getattr(vec, "vector_search_dimensions", None):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "content_vector field has no vector_search_dimensions. "
                    "Pass vector_search_dimensions=1536 on the SearchField."
                ),
            )
        if vec.vector_search_dimensions != EMBED_DIMENSIONS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"content_vector dimensions are {vec.vector_search_dimensions}, "
                    f"expected {EMBED_DIMENSIONS}. text-embedding-3-small returns 1536-dim "
                    f"vectors; a mismatch fails at query time."
                ),
            )
        if not getattr(vec, "vector_search_profile_name", None):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "content_vector field has no vector_search_profile_name set. "
                    "Without a profile, the field is just a numeric array; the index "
                    "cannot run vector queries against it."
                ),
            )

        if not idx.vector_search or not idx.vector_search.algorithms:
            return CheckpointResult(
                status="fail",
                error_reason="Index has no vector_search.algorithms. Declare an HNSW algorithm config.",
            )
        algo_kinds = []
        for a in idx.vector_search.algorithms:
            kind = getattr(a, "kind", None) or type(a).__name__
            algo_kinds.append(str(kind).lower())
        if not any("hnsw" in k for k in algo_kinds):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No HNSW algorithm configuration found. Got: {algo_kinds}. "
                    f"Add an HnswAlgorithmConfiguration to vector_search.algorithms."
                ),
            )

        if not idx.semantic_search or not idx.semantic_search.configurations:
            return CheckpointResult(
                status="fail",
                error_reason="Index has no semantic_search.configurations. Add a SemanticConfiguration.",
            )
        sc = idx.semantic_search.configurations[0]
        pf = getattr(sc, "prioritized_fields", None)
        title_field = getattr(pf, "title_field", None) if pf else None
        content_fields = getattr(pf, "content_fields", None) or [] if pf else []
        if not title_field or getattr(title_field, "field_name", None) != "title":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Semantic configuration does not name 'title' as the title field."
                ),
            )
        if not any(getattr(f, "field_name", None) == "content" for f in content_fields):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Semantic configuration does not include 'content' as a content field."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint pinpoints the exact missing field, profile, or config. A common miss: setting `vector_search_dimensions=1024` (the default for OpenAI's older `text-embedding-ada-002`). For `text-embedding-3-small` the value is 1536.

</details>

<details><summary>Hint 2 (direction)</summary>

One block. Construct `index = SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vector_search, semantic_search=semantic_search)`, then `created_index = search_index_client.create_or_update_index(index)`. All four named pieces (`fields`, `vector_search`, `semantic_search`, and the `INDEX_NAME` string) are already prepared in the cell above. The `vector_search` already has the HNSW algorithm and the profile; the `semantic_search` already has the prioritized fields.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)
created_index = search_index_client.create_or_update_index(index)
```

</details>

**Wallet check.** The index call is a control-plane operation, free. You are still on the hook for the Search hourly meter (a couple cents now) and zero on tokens. Coffee is still ahead by orders of magnitude.

## Task 3: Embed the 50-chunk corpus, upload, and run a hybrid plus semantic-ranked query

The lab ships a 50-chunk seeded corpus inline (no external fetches). Each chunk has a `title`, `content`, and a `source_page`. Your job is to embed every chunk's content with the `text-embedding-3-small` deployment, attach the vector to the chunk, upload all 50 in one batch, then run a hybrid plus semantic-ranked query against a seeded test prompt.

Build it in this order:

1. Construct an `AzureOpenAI` client against the attached AOAI account with `DefaultAzureCredential` (token provider via `bearer_token_provider`).
2. Call `client.embeddings.create(model=EMBED_DEPLOYMENT_NAME, input=[chunk["content"] for chunk in CORPUS_CHUNKS])` to embed all 50 chunks in one batch.
3. Attach each embedding to its chunk and upload via `SearchClient.upload_documents(documents=chunks_with_vectors)`. Wait a few seconds for index commit.
4. Embed the test query, then run `search_client.search(search_text=test_query, vector_queries=[VectorizedQuery(vector=query_vec, k_nearest_neighbors=3, fields="content_vector")], query_type="semantic", semantic_configuration_name="default", top=3)`.
5. Capture the results in `QUERY_RESULTS` so the validator can read scores and reranker scores.

**Why one batch call for embeddings.** OpenAI embeddings accept an `input` array. One round trip for 50 chunks is cheaper than 50 round trips, and Azure's quota system counts requests separately from tokens. The token cost is identical either way; the request count makes the batch path the right one.

**Why hybrid plus semantic.** `search_text=...` invokes BM25 keyword search. `vector_queries=[...]` invokes HNSW vector search. `query_type="semantic"` invokes the semantic ranker over the merged top-50 results. The three components cover three failure modes (exact-match miss, semantic-match miss, top-k ordering noise). The exam tests this exact combination.

In [ ]:
# NBVAL_SKIP
# Task 3: Embed corpus, upload chunks, run hybrid + semantic query.

# A seeded 50-chunk corpus on product-manual content. The chunks intentionally
# include the test-query topic (API key rotation) so the substring assertion
# in the checkpoint can succeed.
CORPUS_CHUNKS = []
for i in range(50):
    page = (i // 5) + 1
    chunk = {
        "id": f"chunk-{i:03d}",
        "title": f"Section {page}.{(i % 5) + 1}",
        "content": "",
        "source_page": page,
    }
    if i == 7:
        chunk["title"] = "Rotating the API key"
        chunk["content"] = (
            "To rotate the API key, sign in to the admin console, open Settings, "
            "select Security, and click Rotate Key. The current key remains valid "
            "for 24 hours so deployments have time to redeploy with the new value. "
            "Rotation is a manual operation; there is no scheduled auto-rotation "
            "for this product as of version 5.2."
        )
    elif i == 12:
        chunk["title"] = "Configuring webhooks"
        chunk["content"] = (
            "Webhook URLs are configured under Settings > Integrations. The "
            "product POSTs a JSON payload to each registered URL on every "
            "qualifying event with a 5-second timeout per delivery attempt."
        )
    elif i == 19:
        chunk["title"] = "Single sign-on with SAML"
        chunk["content"] = (
            "SAML single sign-on is configured per-tenant. The IdP metadata XML "
            "is uploaded under Settings > Authentication. Once enabled, password "
            "login is disabled for non-admin users; admins retain a break-glass "
            "password login."
        )
    else:
        chunk["content"] = (
            f"Section {page}.{(i % 5) + 1} of the product manual. This passage "
            f"covers configuration option {i} and includes references to related "
            f"sections elsewhere in the manual."
        )
    CORPUS_CHUNKS.append(chunk)

print(f"Seeded {len(CORPUS_CHUNKS)} corpus chunks.")

# AzureOpenAI client with DefaultAzureCredential. The bearer_token_provider
# converts the credential into the per-call token function the SDK expects.
from azure.identity import get_bearer_token_provider
token_provider = get_bearer_token_provider(
    AZURE_CREDENTIAL, "https://cognitiveservices.azure.com/.default",
)
aoai_client = AzureOpenAI(
    azure_endpoint=AOAI_ACCOUNT_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2024-08-01-preview",
)

# Embed the 50 chunks in one batch call.
print("Embedding the 50-doc corpus into 1536-dim vectors, this takes 30 to 60 seconds...")
# YOUR CODE: Embed every chunk in one batch via
# embed_response = aoai_client.embeddings.create(
#     model=EMBED_DEPLOYMENT_NAME,
#     input=[chunk["content"] for chunk in CORPUS_CHUNKS],
# )

# Attach each embedding to its chunk.
chunks_with_vectors = []
for chunk, datum in zip(CORPUS_CHUNKS, embed_response.data):
    chunks_with_vectors.append({**chunk, "content_vector": datum.embedding})

# Upload to the index in one batch.
search_data_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AZURE_CREDENTIAL,
)
upload_result = search_data_client.upload_documents(documents=chunks_with_vectors)
print(f"Uploaded {sum(1 for r in upload_result if r.succeeded)} of {len(chunks_with_vectors)} documents.")

# Index commit is async; wait a few seconds for the count to settle.
print("Asking Search Basic to wake up and commit the index, gives it about a minute...")
for _ in range(20):
    count_now = search_data_client.get_document_count()
    if count_now >= len(chunks_with_vectors):
        break
    time.sleep(3)
print(f"Index now reports {search_data_client.get_document_count()} documents.")

# Run a hybrid + semantic query against a seeded test prompt.
test_query = "How do I rotate the API key?"
print()
print(f"Running a hybrid query through the semantic ranker for: {test_query!r}")
query_vec_response = aoai_client.embeddings.create(
    model=EMBED_DEPLOYMENT_NAME, input=[test_query],
)
query_vector = query_vec_response.data[0].embedding

# YOUR CODE: Run the hybrid + semantic-ranked search via
# results_iter = search_data_client.search(
#     search_text=test_query,
#     vector_queries=[VectorizedQuery(
#         vector=query_vector,
#         k_nearest_neighbors=3,
#         fields="content_vector",
#     )],
#     query_type="semantic",
#     semantic_configuration_name="default",
#     top=3,
# )
# Then materialise the iterator: QUERY_RESULTS = list(results_iter).

print(f"Got {len(QUERY_RESULTS)} results back.")
for r in QUERY_RESULTS:
    print(f"  id={r['id']}  score={r['@search.score']:.4f}  "
          f"reranker={r.get('@search.reranker_score', 'n/a')}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Index contains 50 documents. The hybrid + semantic query
# returned 3 results. Every result has both a @search.score and a
# @search.reranker_score (the latter proves the semantic ranker ran). The
# top result's content overlaps with the API-key-rotation chunk.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        sdc = SearchClient(
            endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=AZURE_CREDENTIAL,
        )
        try:
            count = sdc.get_document_count()
        except HttpResponseError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"get_document_count raised HttpResponseError: {e.message}",
            )
        if count != 50:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Index document count is {count}, expected 50. Did the upload "
                    f"complete and the index commit settle?"
                ),
            )

        if not isinstance(QUERY_RESULTS, list) or len(QUERY_RESULTS) < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"QUERY_RESULTS has {len(QUERY_RESULTS) if isinstance(QUERY_RESULTS, list) else 'no'} "
                    f"entries, expected 3. Did you set top=3 and materialise the iterator?"
                ),
            )

        for i, r in enumerate(QUERY_RESULTS[:3], start=1):
            if "@search.score" not in r:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Result {i} has no @search.score field.",
                )
            if "@search.reranker_score" not in r:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Result {i} has no @search.reranker_score. The semantic ranker "
                        f"did not run. Pass query_type='semantic' AND "
                        f"semantic_configuration_name='default' on the search call."
                    ),
                )
            for field in ("id", "title", "content"):
                if not r.get(field):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Result {i} field {field!r} is missing or empty. "
                            f"Did the upload preserve all chunk fields?"
                        ),
                    )

        # Top result should overlap with the API-key-rotation chunk content.
        top_content = (QUERY_RESULTS[0].get("content") or "").lower()
        if not ("api key" in top_content and "rotate" in top_content):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Top result content does not contain 'api key' and 'rotate'. "
                    f"Got: {QUERY_RESULTS[0].get('content')!r:.120}. "
                    f"Check that the embedding model used to embed the corpus matches the "
                    f"one used to embed the query (both should be text-embedding-3-small)."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint will tell you which assertion failed. A `@search.reranker_score` missing means semantic ranking did not run; recheck `query_type="semantic"` AND `semantic_configuration_name="default"`. A document count below 50 means the upload did not finish committing; wait longer (the polling loop in the cell handles this).

</details>

<details><summary>Hint 2 (direction)</summary>

Two calls. Embedding is `embed_response = aoai_client.embeddings.create(model=EMBED_DEPLOYMENT_NAME, input=[chunk["content"] for chunk in CORPUS_CHUNKS])`. The query uses a `VectorizedQuery(vector=query_vector, k_nearest_neighbors=3, fields="content_vector")` passed as a list to `vector_queries=`, alongside `search_text=test_query`, `query_type="semantic"`, `semantic_configuration_name="default"`, and `top=3`. Materialise the iterator with `QUERY_RESULTS = list(results_iter)`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
embed_response = aoai_client.embeddings.create(
    model=EMBED_DEPLOYMENT_NAME,
    input=[chunk["content"] for chunk in CORPUS_CHUNKS],
)

results_iter = search_data_client.search(
    search_text=test_query,
    vector_queries=[VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=3,
        fields="content_vector",
    )],
    query_type="semantic",
    semantic_configuration_name="default",
    top=3,
)
QUERY_RESULTS = list(results_iter)
```

</details>

**Wallet check.** Embedding 50 chunks at roughly 100 tokens each is about 5,000 tokens, which at $0.02 per 1M tokens is $0.0001. The query embedding is another negligible fraction of a penny. Search keeps ticking at about 10 cents per hour. Coffee is now only about 25x ahead. Cleanup matters.

## Task 4: Build the RAG loop and prove every citation resolves to a real indexed chunk

The grounding contract: the model gets the retrieved chunks in its context, gets a system prompt that says "only cite chunks I gave you," and is asked to emit a structured response with a prose answer AND a `citations` array listing the chunk IDs it actually used.

Build it in this order:

1. Re-embed a fresh user question. Run the same hybrid plus semantic-ranked query for top-3 chunks.
2. Build a system prompt that lists each retrieved chunk by `chunk_id` and instructs the model to ONLY cite from that list. Use `response_format={"type": "json_object"}` so the assistant must return parseable JSON.
3. Call `chat.completions.create(model=CHAT_DEPLOYMENT_NAME, messages=[{system}, {user}], response_format=...)` and parse the assistant content as JSON. The shape: `{"answer": "...", "citations": [{"chunk_id": "...", "source_page": ...}]}`.
4. For every citation, look the `chunk_id` up via `search_data_client.get_document(key=chunk_id)`. If it does not resolve, the model hallucinated; fail loudly. Capture the citations array in `RAG_CITATIONS`.

**Why the validator checks lookup, not string match.** The model could emit a citation ID that happens to match a real chunk format (`chunk-019`) but is not in the top-3 retrieved set. The lookup confirms the ID exists in the index; the system prompt instructs the model to ONLY cite the retrieved IDs. The combination prevents hallucinated citations.

**Why JSON mode.** Without `response_format={"type": "json_object"}`, GPT-4o-mini will sometimes return Markdown around the JSON. Parsing breaks. JSON mode forces a strict JSON output.

In [ ]:
# NBVAL_SKIP
# Task 4: RAG loop with citation extraction and resolution.

user_question = "Walk me through how to rotate the API key, and link the section."
print(f"User question: {user_question!r}")

# Re-embed and re-run the query.
q_emb = aoai_client.embeddings.create(model=EMBED_DEPLOYMENT_NAME, input=[user_question])
q_vec = q_emb.data[0].embedding
retrieval = list(search_data_client.search(
    search_text=user_question,
    vector_queries=[VectorizedQuery(
        vector=q_vec, k_nearest_neighbors=3, fields="content_vector",
    )],
    query_type="semantic",
    semantic_configuration_name="default",
    top=3,
))
print(f"Retrieved {len(retrieval)} chunks for grounding.")

# Build the chunk-grounded system prompt. Listing every chunk by ID and
# instructing the model to ONLY cite from that list is the prompt-level
# defence against hallucinated citations.
context_lines = []
for r in retrieval:
    context_lines.append(
        f"chunk_id={r['id']}  source_page={r['source_page']}  title={r['title']}\n"
        f"  content: {r['content']}"
    )
retrieved_context = "\n\n".join(context_lines)
allowed_ids = [r["id"] for r in retrieval]

system_prompt = (
    "You are a documentation assistant. You answer the user's question grounded "
    "in the provided context chunks. You must cite at least one chunk_id from the "
    "allowed list. NEVER cite a chunk_id outside that list, even if it sounds "
    "plausible. If the context does not answer the question, say so and cite the "
    "closest chunk as the source you considered. Return JSON exactly in this shape:\n"
    '{"answer": "<prose answer>", "citations": [{"chunk_id": "...", "source_page": N}]}\n\n'
    f"Allowed chunk_ids: {allowed_ids}\n\n"
    f"Context:\n{retrieved_context}"
)

# YOUR CODE: Call the chat completion via
# rag_response = aoai_client.chat.completions.create(
#     model=CHAT_DEPLOYMENT_NAME,
#     messages=[
#         {"role": "system", "content": system_prompt},
#         {"role": "user", "content": user_question},
#     ],
#     response_format={"type": "json_object"},
# )

raw_content = rag_response.choices[0].message.content or "{}"
parsed = json.loads(raw_content)
RAG_ANSWER = parsed.get("answer", "")
RAG_CITATIONS = parsed.get("citations", [])

print()
print("Assistant answer:")
print(RAG_ANSWER)
print()
print("Cited chunk_ids:")
for c in RAG_CITATIONS:
    print(f"  - {c.get('chunk_id')!r} (page {c.get('source_page')})")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: RAG_CITATIONS is non-empty; every chunk_id resolves via
# search_data_client.get_document; every cited chunk_id is also in the
# retrieved top-3 (no out-of-retrieval citations).

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        if not isinstance(RAG_CITATIONS, list) or len(RAG_CITATIONS) < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"RAG_CITATIONS is empty. The model must cite at least one chunk_id. "
                    f"Got: {RAG_CITATIONS!r}"
                ),
            )

        sdc = SearchClient(
            endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=AZURE_CREDENTIAL,
        )
        for c in RAG_CITATIONS:
            chunk_id = c.get("chunk_id") if isinstance(c, dict) else None
            if not chunk_id:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"A citation is missing chunk_id. Got: {c!r}. "
                        f"The expected shape is {{'chunk_id': '...', 'source_page': N}}."
                    ),
                )
            try:
                sdc.get_document(key=chunk_id)
            except ResourceNotFoundError:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Citation {chunk_id!r} does not resolve to an indexed chunk. "
                        f"The model hallucinated a chunk_id. Tighten the system prompt: "
                        f"list every retrieved chunk_id and instruct ONLY-cite-from-that-list."
                    ),
                )
            except HttpResponseError as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"get_document raised HttpResponseError: {e.message}",
                )

        if not (RAG_ANSWER or "").strip():
            return CheckpointResult(
                status="fail",
                error_reason="RAG_ANSWER is empty. The model returned a citations array but no prose answer.",
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If the citation does not resolve via `get_document`, the model hallucinated a chunk_id. Tighten the system prompt: it must list every retrieved chunk_id and instruct the model to ONLY cite from that list. If `RAG_CITATIONS` is empty, the model returned an answer with no citation block; require citations explicitly in the system prompt.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `rag_response = aoai_client.chat.completions.create(model=CHAT_DEPLOYMENT_NAME, messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_question}], response_format={"type": "json_object"})`. The system prompt is already built. JSON mode prevents Markdown fences around the JSON. Parse `rag_response.choices[0].message.content` with `json.loads`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
rag_response = aoai_client.chat.completions.create(
    model=CHAT_DEPLOYMENT_NAME,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question},
    ],
    response_format={"type": "json_object"},
)
```

</details>

**Wallet check.** The RAG turn at ~1,500 input + ~400 output tokens runs about $0.00023 + $0.00024 = under a tenth of a cent. Embeddings for the user question are negligible. Search keeps ticking at about 10 cents per hour. Total session damage: a few cents at most, dominated by the Search service. Coffee remains comfortably ahead, assuming you actually run cleanup.

## Cleanup

Time to tear it all down. **The Search service is the cost-critical resource in this lab.** The cell below runs the manifest in reverse-creation order (index, then search service, then deployments, then project, hub, resource group), then verifies via Resource Graph that nothing tagged with this lab's slug is left. Re-running is safe. If cleanup fails on any resource, `sys.exit(1)` fires so the companion panel marks the session dirty and you have a chance to fix it before walking away.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. The Search service is the only critical resource; deleting it stops
# the $0.10/hour meter.

import sys

# Resolve AOAI_ACCOUNT_NAME into the deployment CLI strings now that it has
# been discovered.
for r in CLEANUP_MANIFEST:
    if AOAI_ACCOUNT_NAME and "<attached-aoai-account>" in (r.cli_delete_command or ""):
        r.cli_delete_command = r.cli_delete_command.replace(
            "<attached-aoai-account>", AOAI_ACCOUNT_NAME,
        )

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type == "ai_search_service")
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_failed = sum(1 for r in CLEANUP_MANIFEST if r.type == "ai_search_service" and r.id in failed_ids)
standard_failed = len(failed_ids) - critical_failed
critical_destroyed = critical_total - critical_failed
standard_destroyed = standard_total - standard_failed
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Azure subscription may still incur charges. Resolve before closing this notebook.")
if critical_failed > 0:
    print()
    print("CRITICAL: an Azure AI Search Basic service is still alive.")
    print(f"That meter is ticking at ${SEARCH_BASIC_HOURLY_USD:.2f}/hour ($73.73/month).")
    print(f"Run manually: az search service delete --resource-group {RESOURCE_GROUP} "
          f"--name {SEARCH_SERVICE_NAME} --yes")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.20 to $0.30 depending on session length.** Search Basic at $0.10/hour dominates; GPT-4o-mini and `text-embedding-3-small` together cost fractions of a cent. The Foundry hub, project, and Standard deployments carry no hourly fee at zero traffic. Check Azure Cost Management in 24 hours to confirm the exact number (Cost Management has a 24-hour-plus data lag, same as AWS Cost Explorer). If you see Search Basic in tomorrow's cost view, run `az search service list -o table` and verify the service is gone.

## Reflection

These are not graded. They are for you.

1. The Checkpoint 4 validator confirmed every citation's `chunk_id` resolved to a real indexed chunk. That guards against fabricated citations but not against the model citing the right chunk and then writing a paragraph that contradicts it. Walk through what additional evaluation you would add to catch grounding-quality drift, and which evaluator from `azure-ai-evaluation` you would use.

2. Your team is choosing between Basic ($73.73/month, 15 GB, 15 indexes) and Standard S1 ($245.28/month, 160 GB) for the production RAG service. What signals would push you to S1? What would you measure during the Basic-tier pilot to know the answer with confidence?

## Exam-Style Practice

**Question 1 (Domain 5, retrieval mode selection):**

A RAG application built on Azure AI Search runs vector-only queries today. Users frequently search by exact product SKU (for example, `SKU-A123-X`) and complain that the vector search returns semantically similar but wrong SKUs. The team wants to fix this without sacrificing the semantic match quality on natural-language queries. Which retrieval configuration fits?

A. Switch to keyword-only (BM25) search; the vector embeddings are causing the SKU misses.

B. Use hybrid retrieval (vector plus BM25) without the semantic ranker; the BM25 signal pulls exact SKU matches to the top while vector preserves semantic match.

C. Use hybrid retrieval (vector plus BM25) with the semantic ranker enabled; BM25 pulls exact SKUs, vector covers semantic match, and the semantic ranker reorders by query intent.

D. Disable embeddings and rely on the semantic ranker alone for both query types.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: keyword-only loses semantic match on natural-language queries. The team explicitly does not want to lose that.
- B is partially right but misses the semantic ranker, which is the AI-103-explicit pattern for production RAG. Hybrid alone retrieves the right candidates but the top-k ordering can be noisy.
- C is correct: this is the documented "vector plus hybrid plus semantic ranker" production RAG pattern from the AI-103 study guide. Each component covers a failure mode of the others.
- D is wrong: the semantic ranker is a reranker that operates on top-50 candidates returned by retrieval, not a primary retrieval mode. Disabling embeddings leaves the ranker with no semantic vectors to rerank.

</details>

**Question 2 (Domain 2 plus 5, grounding prompt design):**

A RAG chat app's responses occasionally contain factual claims that are not in the retrieved chunks (a "hallucinated citation" where the citation ID is real but the cited content does not actually support the claim). The team wants to reduce these. Which mitigation has the highest expected impact?

A. Increase the temperature on the GPT-4o-mini deployment to 0.8 to encourage more creative answers.

B. Add an "if you do not know, say so" instruction to the system prompt and reduce temperature to 0.2.

C. Switch to GPT-4o instead of GPT-4o-mini; the larger model hallucinates less.

D. Add a groundedness evaluator to the application that re-scores each response against the retrieved chunks and either rewrites or rejects ungrounded answers before returning them to the user.

<details><summary>Show answer</summary>

**Correct: D.**

- A is wrong: higher temperature increases hallucination, not reduces it.
- B helps marginally but is a prompt-level mitigation only. The model will still hallucinate when the retrieved chunks are ambiguous. Reducing temperature alone is not the highest-impact lever.
- C is partial: GPT-4o has somewhat lower hallucination rates than GPT-4o-mini, but the question asks about hallucinated citations specifically, which is a retrieval-grounding problem, not a model-quality problem.
- D is correct: the `GroundednessEvaluator` from `azure-ai-evaluation` is purpose-built for this. It re-scores each response against its retrieved context and exposes a 1-to-5 groundedness score. Applications can reject or rewrite below-threshold responses. This is the documented AI-103 mitigation for ungrounded RAG.

</details>